<a href="https://colab.research.google.com/github/xuebingwu/kpLogo/blob/master/kplogo_colab_form.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# *k*pLogo: *k*-mer probability logo

Run the [kpLogo web server](https://kplogo.wi.mit.edu/) interactively in Google Colab. This notebook uses the same kpLogo program (version 1.0) and builds the identical command line, so results match the web server.

**How to use**
1. **Runtime > Run all** — the first run installs the plotting tools and downloads the kpLogo binary (~1 min).
2. Open the **kpLogo Job Form** cell (click the form icon on its left if the fields are hidden), fill in section 1, adjust the other sections if needed.
3. **Run the form cell** — figures appear below it and all output files are bundled into a downloadable zip. To run another job, edit the fields and run the cell again.

**Notes**
- For more than a couple of short sequences, set *foreground source* to **upload a file** (FASTA, raw, or tabular format — see the [manual](https://kplogo.wi.mit.edu/manual.html)). A file chooser opens when you run the cell.
- The web server's optional email notification is not available here; results are downloaded directly instead.
- The web form's *predefined Markov model* background (hg/ht/hp/mg/mt/mp) is not implemented by the web server either, so it is omitted here.
- Output files live in the Colab workspace and are deleted when the runtime resets — download anything you want to keep.

## Job Form

The form and the execution share one cell: edit the fields, run the cell, get the figures. Defaults match the web server. Full documentation: [manual](https://kplogo.wi.mit.edu/manual.html).

In [ ]:
#@title Setup (run once) { display-mode: "form" }
import os, re, shutil, subprocess, sys

# kpLogo converts its EPS figures by calling ps2pdf (ghostscript) and convert
# (imagemagick); make sure both are installed and usable
subprocess.run("apt-get update -qq > /dev/null 2>&1", shell=True)
r = subprocess.run("apt-get install -y -qq ghostscript imagemagick > /dev/null 2>&1", shell=True)
for tool in ["ps2pdf", "convert", "gs"]:
    if shutil.which(tool) is None:
        print("\033[91mWARNING: %s is not available - logo figures may not be converted to PDF/PNG.\033[0m" % tool)

# some distros ship ImageMagick with a security policy that forbids reading
# EPS/PS and writing PDF - re-enable the coders kpLogo needs
for pol in ["/etc/ImageMagick-6/policy.xml", "/etc/ImageMagick/policy.xml"]:
    if os.path.exists(pol):
        p = open(pol).read()
        p2 = re.sub(r'<policy domain="coder" rights="none" pattern="(?:EPS|EPI|EPHE|PS|PS2|PS3|PDF|PDFWRITE)"\s*/>', "", p)
        if p2 != p:
            open(pol, "w").write(p2)
            print("Re-enabled EPS/PDF coders in " + pol)

if not os.path.exists("kpLogo"):
    print("Downloading kpLogo binary ...")
    r = subprocess.run("curl -sL -o kpLogo "
        "https://raw.githubusercontent.com/xuebingwu/kpLogo/master/bin/kpLogo", shell=True)
    if r.returncode != 0 or not os.path.exists("kpLogo") or os.path.getsize("kpLogo") < 100000:
        print("Direct download failed; falling back to cloning the repository ...")
        subprocess.run("git clone --depth 1 https://github.com/xuebingwu/kpLogo.git", shell=True, check=True)
        os.replace("kpLogo/bin/kpLogo", "kpLogo")
        subprocess.run("rm -rf kpLogo_repo", shell=True)
    os.chmod("kpLogo", 0o755)

KPLOGO_BIN = os.path.abspath("kpLogo")

subprocess.run("./kpLogo 2>&1 | head -3", shell=True)
print("Setup complete. kpLogo binary: " + KPLOGO_BIN)

In [ ]:

#@markdown ### 1. Input sequences
foreground_source = "paste sequences" #@param ["paste sequences", "upload a file"]
foreground_sequences = "" #@param {type:"string"}
#@markdown Sequences to paste (if *paste sequences*). Separate sequences with line breaks or with the two characters `\n`. For real datasets use *upload a file* (FASTA / raw / tabular).

inputtype = "Binomial test (default)" #@param ["Binomial test (default)", "Wilcoxon rank-sum test (ranked input)", "t-test on weights (weighted input)"]
col_seq = 1 #@param {type:"integer"}
col_weight = 2 #@param {type:"integer"}
#@markdown Tabular input only: column with the sequence, column with the weight/rank.

region_first = 1 #@param {type:"integer"}
region_last = 0 #@param {type:"integer"}
#@markdown Analyze positions region_first..region_last only (0 or negative = counted from the end).

select_pkmers = "" #@param {type:"string"}
remove_pkmers = "" #@param {type:"string"}
#@markdown Optional comma-separated k-mers to keep / drop, e.g. `CNNC:47:0,UGCA:30:1`.

#@markdown ### 2. Alphabet and k-mers
alphabet = "DNA" #@param ["DNA", "Protein", "Other"]
other_alphabet = "" #@param {type:"string"}
#@markdown If *Other*: the allowed residues, e.g. `ACGTN` or `ACDEFGHIKLMNPQRSTVWY`.

kmer_length = "All k-mers up to length 4 (default)" #@param ["All k-mers up to length 6", "All k-mers up to length 5", "All k-mers up to length 4 (default)", "All k-mers up to length 3", "All k-mers up to length 2", "Fixed length 6", "Fixed length 5", "Fixed length 4", "Fixed length 3", "Fixed length 2", "Fixed length 1"]
degenerate = "Gapped k-mers (default)" #@param ["Gapped k-mers (default)", "All degenerate nucleotides", "Custom set", "None"]
degenerate_alphabet = "" #@param {type:"string"}
#@markdown If *Custom set*: allowed degenerate symbols (subset of `ACGTRYMKWSBDHVN`). Ignored unless the alphabet is DNA.

shift = "No shift (default)" #@param ["No shift (default)", "Max shift 1", "Max shift 2", "Max shift 3", "Max shift 4", "Max shift 5", "Max shift 6", "Consider shifts 0-1", "Consider shifts 0-2", "Consider shifts 0-3", "Consider shifts 0-4", "Consider shifts 0-5", "Consider shifts 0-6"]

#@markdown ### 3. Background model (ignored for ranked / weighted input)
background = "Same k-mer at other positions (default)" #@param ["Same k-mer at other positions (default)", "Background sequence file", "Shuffled input sequences", "Markov model trained on foreground", "Markov model trained on background file", "Markov model given as text"]
background_source = "paste sequences" #@param ["paste sequences", "upload a file"]
background_sequences = "" #@param {type:"string"}
#@markdown Background sequences (used by *Background sequence file* and *Markov model trained on background file*).

shuffle_n = 100 #@param {type:"integer"}
shuffle_m = "2" #@param ["1", "2", "3", "4"] {type:"raw"}
markov_foreground_order = "1" #@param ["0", "1", "2"] {type:"raw"}
markov_background_order = "1" #@param ["0", "1", "2"] {type:"raw"}
markov_string = "A:0.25,C:0.25,G:0.25,T:0.25" #@param {type:"string"}

#@markdown ### 4. Statistics and cutoffs
mincount = 0.01 #@param {type:"number"}
#@markdown Minimum count for a k-mer to be included (<1 = fraction of sequences).

pseudo = 1 #@param {type:"number"}
p = 0.01 #@param {type:"number"}
#@markdown p-value cut-off (Bonferroni-corrected).

maxFrac = 0.75 #@param {type:"number"}
#@markdown Fix a position with a specific residue if it occurs in more than this fraction of sequences.

small_sample = False #@param {type:"boolean"}
#@markdown Correct for small sample size.

#@markdown ### 5. Plotting and output
colorblind = False #@param {type:"boolean"}
last_residue = False #@param {type:"boolean"}
#@markdown Position a k-mer by its last (instead of first) residue.

startPos = 1 #@param {type:"integer"}
#@markdown Re-number this position as 1 (positions before it become negative).

plottype = "Raw p-value (default)" #@param ["Raw p-value (default)", "Bonferroni-corrected p-value", "Test statistic"]
stack_order = "By frequency (default)" #@param ["By frequency (default)", "Reverse frequency", "Alphabetical"]
jobname = "my kpLogo job" #@param {type:"string"}
# ===== kpLogo execution - no need to edit below this line =====
import os, re, subprocess, glob, zipfile

# locate the kpLogo binary (set by the Setup cell)
KPLOGO_BIN = globals().get("KPLOGO_BIN")
if not KPLOGO_BIN or not os.path.exists(KPLOGO_BIN):
    for cand in ["/content/kpLogo", "../kpLogo", "./kpLogo"]:
        if os.path.exists(cand):
            KPLOGO_BIN = cand
            break
    else:
        raise SystemExit("kpLogo binary not found - please run the Setup cell first.")

# same sanitization as the web server (web/submit.php)
def scrub_alphanum(s):
    if s is None: s = ""
    s = re.sub(r"\.+", ".", s)                       # all repeating dots to single dots
    s = re.sub(r"[^@0-9a-zA-Z\-_:,. ]", "", s)        # keep alphanum, comma, period, at, hyphen
    return s

def die(msg):
    print("\033[91m" + msg + "\033[0m")
    raise SystemExit(1)

def upload_file(prompt):
    # asks the user for a file; works in Colab (browser file chooser) and in
    # local Jupyter (path prompt)
    print(prompt)
    print("(a file chooser will open below - select your file and wait until the upload finishes)")
    try:
        from google.colab import files
        up = files.upload()
    except ImportError:  # local Jupyter fallback
        fn = input("Local Jupyter detected - enter the path of your file: ").strip()
        with open(fn, "rb") as f:
            up = {os.path.basename(fn): f.read()}
    if not up:
        die("No file was uploaded - please run the cell again and choose a file.")
    name = list(up.keys())[0]
    data = up[name]
    if len(data) == 0:
        die("The uploaded file '%s' is empty - please check it and try again." % name)
    if len(up) > 1:
        print("Note: several files were chosen; using the first one (%s)." % name)
    print("Received: %s (%d bytes)" % (name, len(data)))
    return data

# ---------- friendly form labels -> kpLogo command-line values ----------
# (raw CLI values are also accepted, so old notebooks keep working)
INPUTTYPE = {"Binomial test (default)": "", "Wilcoxon rank-sum test (ranked input)": "-ranked",
             "t-test on weights (weighted input)": "-weighted"}
ALPHABET = {"DNA": "dna", "Protein": "protein", "Other": "other"}
KMER = {"All k-mers up to length 6": "-max_k 6", "All k-mers up to length 5": "-max_k 5",
        "All k-mers up to length 4 (default)": "-max_k 4", "All k-mers up to length 3": "-max_k 3",
        "All k-mers up to length 2": "-max_k 2", "Fixed length 6": "-k 6", "Fixed length 5": "-k 5",
        "Fixed length 4": "-k 4", "Fixed length 3": "-k 3", "Fixed length 2": "-k 2", "Fixed length 1": "-k 1"}
DEGEN = {"Gapped k-mers (default)": "-gapped", "All degenerate nucleotides": "-degenerate all",
         "Custom set": "other", "None": "(none)"}
SHIFT = {"No shift (default)": "-shift 0", "Max shift 1": "-shift 1", "Max shift 2": "-shift 2",
         "Max shift 3": "-shift 3", "Max shift 4": "-shift 4", "Max shift 5": "-shift 5",
         "Max shift 6": "-shift 6", "Consider shifts 0-1": "-max_shift 1", "Consider shifts 0-2": "-max_shift 2",
         "Consider shifts 0-3": "-max_shift 3", "Consider shifts 0-4": "-max_shift 4",
         "Consider shifts 0-5": "-max_shift 5", "Consider shifts 0-6": "-max_shift 6"}
BACKGROUND = {"Same k-mer at other positions (default)": "", "Background sequence file": "bgfile",
              "Shuffled input sequences": "shuffle", "Markov model trained on foreground": "markov_foreground",
              "Markov model trained on background file": "markov_background",
              "Markov model given as text": "markov_string"}
PLOTTYPE = {"Raw p-value (default)": "-plot p", "Bonferroni-corrected p-value": "-plot b",
            "Test statistic": "-plot s"}
STACK = {"By frequency (default)": "-stack_order 1", "Reverse frequency": "-stack_order -1",
         "Alphabetical": "-stack_order 0"}

it        = INPUTTYPE.get(inputtype, inputtype)
alphabet_key = ALPHABET.get(alphabet, alphabet)
kmer_opt  = KMER.get(kmer_length, kmer_length)
deg_key   = DEGEN.get(degenerate, degenerate)
shift_opt = SHIFT.get(shift, shift)
bg_key    = BACKGROUND.get(background, background)
plot_opt  = PLOTTYPE.get(plottype, plottype)
stack_opt = STACK.get(stack_order, stack_order)

jobname = scrub_alphanum(jobname) or "kpLogo job"
jobdir = "kpLogo_job_" + re.sub(r"[^0-9a-zA-Z\-_]", "_", jobname)
os.makedirs(jobdir, exist_ok=True)
os.chdir(jobdir)

# ---------- foreground input ----------
if foreground_source == "paste sequences":
    seqs = foreground_sequences.replace("\\n", "\n")
    if len(seqs.strip()) == 0:
        die("Please paste your sequences or choose 'upload a file' in the form.")
    with open("kpLogo.input.txt", "w") as f:
        f.write(seqs)
else:
    with open("kpLogo.input.txt", "wb") as f:
        f.write(upload_file("Please upload your foreground sequence file (FASTA / raw / tabular):"))

# ---------- background input ----------
bg = ""
if bg_key == "bgfile":
    bg = " -bgfile kpLogo.background.txt "
elif bg_key == "markov_foreground":
    bg = " -markov %s " % markov_foreground_order
elif bg_key == "shuffle":
    bg = " -shuffle %s,%s " % (shuffle_n, shuffle_m)
elif bg_key == "markov_background":
    bg = " -markov %s -bgfile kpLogo.background.txt " % markov_background_order
elif bg_key == "markov_string":
    bg = " -markov %s" % scrub_alphanum(markov_string)

if bg_key in ("bgfile", "markov_background"):
    if background_source == "paste sequences":
        bseqs = background_sequences.replace("\\n", "\n")
        if len(bseqs.strip()) == 0:
            die("Please paste your background sequences or choose 'upload a file' in the form.")
        with open("kpLogo.background.txt", "w") as f:
            f.write(bseqs)
    else:
        with open("kpLogo.background.txt", "wb") as f:
            f.write(upload_file("Please upload your background sequence file:"))

# ---------- options (same mapping as web/submit.php) ----------
alphabet_v = scrub_alphanum(alphabet_key)
if alphabet_v == "other":
    alphabet_v = scrub_alphanum(other_alphabet)
    if alphabet_v == "":
        die("Please enter the allowed residues for the 'Other' alphabet.")

deg = deg_key
if deg == "other":
    da = scrub_alphanum(degenerate_alphabet)
    if da == "":
        die("Please enter degenerate symbols!")
    deg = " -degenerate %s " % da
elif deg == "(none)":
    deg = ""
if alphabet_v != "dna":
    deg = ""  # same rule as the web server

sel = (" -select " + scrub_alphanum(select_pkmers)) if scrub_alphanum(select_pkmers) != "" else ""
rem = (" -remove " + scrub_alphanum(remove_pkmers)) if scrub_alphanum(remove_pkmers) != "" else ""

mincount_v = mincount if mincount >= 0 else 0  # same clamp as the web server

# ---------- command (identical construction to web/submit.php) ----------
cmd = (KPLOGO_BIN + " kpLogo.input.txt -o kpLogo.output %s -seq %s -weight %s -alphabet %s  %s %s %s "
       "-startPos %s %s %s -minCount %s -pseudo %s -region %s,%s %s %s %s %s -pc %s %s %s -fix %s"
       ) % (it, scrub_alphanum(str(col_seq)), scrub_alphanum(str(col_weight)), alphabet_v,
            kmer_opt, shift_opt, bg, scrub_alphanum(str(startPos)), deg,
            "-colorblind" if colorblind else "",
            scrub_alphanum(str(mincount_v)), scrub_alphanum(str(pseudo)),
            scrub_alphanum(str(region_first)), scrub_alphanum(str(region_last)),
            sel, rem, plot_opt,
            "-small_sample" if small_sample else "",
            scrub_alphanum(str(p)),
            "-last_residue" if last_residue else "",
            stack_opt, scrub_alphanum(str(maxFrac)))

print("Job '%s' starting - alphabet: %s | k-mers: %s | background: %s" %
      (jobname, alphabet_v, kmer_opt, bg_key or "default (other positions)"))
print("Running:", cmd.replace(KPLOGO_BIN, "kpLogo"))
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
with open("log", "w") as f:
    f.write(r.stdout + r.stderr)

if os.path.exists("exit_with_error"):
    print("\033[91mExit due to error!\033[0m")
    print(r.stdout + r.stderr)
    raise SystemExit(1)

# ---------- make sure every EPS figure was converted to PDF and PNG ----------
# kpLogo does this itself by calling ps2pdf and convert; if either tool was
# missing at run time the conversion fails silently - redo it here and report.
# Note: when its own conversion step fails, kpLogo leaves the EPS with corrupt
# (uninitialized) coordinates and bounding box; such a figure can only be
# fixed by re-running kpLogo once the plotting tools work.
def run_tool(args):
    try:
        return subprocess.run(args, capture_output=True, text=True)
    except FileNotFoundError:
        class R: returncode, stdout, stderr = 127, "", "tool not found: " + args[0]
        return R()

def bbox_is_corrupt(eps):
    try:
        with open(eps, "r", errors="ignore") as f:
            for line in f:
                if line.startswith("%%BoundingBox:"):
                    vals = [float(v) for v in line.split(":")[1].split()]
                    return any(abs(v) > 1e6 for v in vals)
    except Exception:
        return True
    return False

def convert_figures():
    n_missing = 0
    for eps in sorted(glob.glob("kpLogo.output*.eps")):
        if bbox_is_corrupt(eps):
            n_missing += 1  # unrecoverable; fixed by re-running kpLogo below
            continue
        base = eps[:-4]
        if not os.path.exists(base + ".pdf"):
            c = run_tool(["ps2pdf", "-dEPSCrop", eps, base + ".pdf"])
            if os.path.exists(base + ".pdf"):
                print("Converted " + eps + " -> PDF (was missing)")
            else:
                n_missing += 1
                print("\\033[91mCould not convert %s to PDF: %s\\033[0m" %
                      (eps, (c.stderr or c.stdout or "ps2pdf not found").strip()[:200]))
        if not os.path.exists(base + ".png"):
            c = run_tool(["convert", eps, base + ".png"])
            if not os.path.exists(base + ".png"):
                c = run_tool(["gs", "-dSAFER", "-dBATCH", "-dNOPAUSE",
                              "-sDEVICE=pngalpha", "-r150", "-dEPSCrop",
                              "-sOutputFile=" + base + ".png", eps])
            if os.path.exists(base + ".png"):
                print("Converted " + eps + " -> PNG (was missing)")
            else:
                n_missing += 1
                print("\\033[91mCould not convert %s to PNG: %s\\033[0m" %
                      (eps, (c.stderr or c.stdout or "convert/gs not found").strip()[:200]))
    return n_missing

n_missing = convert_figures()
corrupt = [e for e in glob.glob("kpLogo.output*.eps") if bbox_is_corrupt(e)]
if corrupt:
    print("Re-running kpLogo: %d figure(s) were left corrupt because the plotting tools were unavailable during the first pass." % len(corrupt))
    subprocess.run(cmd, shell=True, capture_output=True, text=True)
    n_missing = convert_figures()
if n_missing:
    print("\\033[91m%d figure conversion(s) still failed - run the Setup cell (it installs ghostscript and imagemagick), then run this cell again.\\033[0m" % n_missing)

# ---------- display results ----------
try:
    from IPython.display import Image, display
    figures = [("Frequency Logo", "kpLogo.output.freq.png"),
               ("Information Content Logo", "kpLogo.output.info.png"),
               ("k-mer Logo - most significant k-mer at each position", "kpLogo.output.most.significant.each.position.png"),
               ("Probability Logo", "kpLogo.output.png")]
    figures = [(t, p) for t, p in figures if os.path.exists(p)]
    if figures:
        print("\nResults - %d figure(s):" % len(figures))
        for i, (title, png) in enumerate(figures, 1):
            print("\n%d. %s" % (i, title))
            display(Image(png, width=750))
    else:
        print("No logo figures were produced (no significant k-mers found) - see the screen output below.")
except ImportError:
    pass

# ---------- tabular output and log ----------
tables = [t for t in ["kpLogo.output.most.significant.each.position.txt",
                      "kpLogo.output.pass.p.cutoff.txt"] if os.path.exists(t)]
if tables:
    print("\nTabular output:")
    for t in tables:
        print("  " + t)

print("\nScreen output:")
print(r.stdout[-3000:] if len(r.stdout) > 3000 else r.stdout)

# ---------- bundle for download ----------
out = jobname + ".zip"
with zipfile.ZipFile(out, "w") as z:
    for f in sorted(glob.glob("kpLogo.output.*")) + ["kpLogo.input.txt", "log"]:
        if os.path.exists(f):
            z.write(f)
print("\nAll output files were saved to " + out + " (in " + os.getcwd() + ")")
try:
    from google.colab import files
    files.download(out)
except ImportError:
    pass